# Chapter 17 — Multilayer Feedforward Neural Networks I

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch17_multilayer_feedforward_neural_networks_i/ch17_notebook.ipynb)

Companion notebook for **Chapter 17** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- A neuron computes a weighted sum of its inputs and then applies a nonlinear activation function.
- The activation choice matters: sigmoid and tanh saturate at the extremes, while ReLU has gradient 1 for all positive inputs.
- Softmax converts output scores into class probabilities that sum to exactly 1.
- A single perceptron can only draw a straight boundary, which is why it fails on XOR.
- Adding one hidden layer solves XOR — nonlinearity in the hidden layer is what buys the extra power.

## Example 1: Activation Functions Compared

Before building a neural network, you need to understand what each activation function does. This example computes and compares Logistic, Tanh, ReLU, and Softmax over a range of inputs. The key insight is in the derivative table at the bottom: for logistic at a=±4, the gradient is only 0.0177 — nearly zero. In a deep network with 10 sigmoid layers, the gradient shrinks by 0.0177^10 ≈ 10^-17 by the time it reaches the first layer. That's the vanishing gradient problem. ReLU has a derivative of exactly 1 for all positive inputs — no shrinkage.

In [1]:
import numpy as np

def logistic(a): return 1 / (1 + np.exp(-a))
def tanh(a):     return np.tanh(a)
def relu(a):     return np.maximum(0, a)
def softmax(v):  e = np.exp(v - v.max()); return e / e.sum()

print("=== Activation Functions ===\n")
print(f"{'Input a':>10} {'Logistic':>10} {'Tanh':>10} {'ReLU':>10}")
print(f"{'':>10} {'(0,1)':>10} {'(-1,1)':>10} {'[0,inf)':>10}")
print("-" * 44)
for a in np.linspace(-5, 5, 11):
    print(f"{a:>10.1f} {logistic(a):>10.4f} {tanh(a):>10.4f} {relu(a):>10.4f}")

print()
print("=== Softmax: 4-class probability output ===\n")
raw = np.array([2.0, 1.0, 0.1, -1.5])
probs = softmax(raw)
classes = ['Dog', 'Cat', 'Bird', 'Fish']
print(f"{'Class':<8} {'Raw':>8} {'Prob':>8}")
for cls, r, p in zip(classes, raw, probs):
    print(f"  {cls:<6} {r:>8.1f} {p:>8.4f}  {'X'*int(p*30)}")
print(f"Sum of probs: {probs.sum():.4f}  (always 1.0)")

print()
print("=== Logistic Derivative (used in backprop) ===\n")
print(f"{'Input a':>8} {'f(a)':>10} {'df/da':>10}  Note")
print("-" * 55)
for a in [-4, -2, 0, 2, 4]:
    fa = logistic(a); dfa = fa*(1-fa)
    note = "SATURATED (near-zero gradient!)" if abs(a)>=3 else "Healthy gradient"
    print(f"{a:>8} {fa:>10.4f} {dfa:>10.4f}  {note}")

=== Activation Functions ===

   Input a   Logistic       Tanh       ReLU
                (0,1)     (-1,1)    [0,inf)
--------------------------------------------
      -5.0     0.0067    -0.9999     0.0000
      -4.0     0.0180    -0.9993     0.0000
      -3.0     0.0474    -0.9951     0.0000
      -2.0     0.1192    -0.9640     0.0000
      -1.0     0.2689    -0.7616     0.0000
       0.0     0.5000     0.0000     0.0000
       1.0     0.7311     0.7616     1.0000
       2.0     0.8808     0.9640     2.0000
       3.0     0.9526     0.9951     3.0000
       4.0     0.9820     0.9993     4.0000
       5.0     0.9933     0.9999     5.0000

=== Softmax: 4-class probability output ===

Class         Raw     Prob
  Dog         2.0   0.6461  XXXXXXXXXXXXXXXXXXX
  Cat         1.0   0.2377  XXXXXXX
  Bird        0.1   0.0966  XX
  Fish       -1.5   0.0195  
Sum of probs: 1.0000  (always 1.0)

=== Logistic Derivative (used in backprop) ===

 Input a       f(a)      df/da  Note
---------------

## Example 2: Perceptron Learning Algorithm from Scratch

We build the perceptron learning rule exactly as described in the overview. The dataset has two linearly separable classes: class +1 at (1,1), (2,1), (1,2) and class -1 at (-1,-1), (-2,-1), (-1,-2). We add a bias term by prepending 1 to each input vector. The algorithm starts with zero weights and updates them only when a misclassification happens. The training trace shows exactly which example caused an update and what the new weights are. This converges in just 2 epochs, and the final verification confirms all 6 examples are classified correctly.

In [2]:
import numpy as np

# Augmented inputs [bias=1, x1, x2]
X = np.array([[1,1,1],[1,2,1],[1,1,2],
              [1,-1,-1],[1,-2,-1],[1,-1,-2]])
y = np.array([+1,+1,+1,-1,-1,-1])
w   = np.zeros(3)
eta = 0.5
print("=== Perceptron Learning: Step-by-Step ===\n")
print(f"Initial weights: {w}")
print(f"Learning rate η: {eta}\n")
print(f"{'Epoch':>6} {'Ex':>4} {'True':>6} {'w.x':>8} {'Pred':>6}  New weights")
print("-" * 65)
for epoch in range(1, 15):
    errors = 0
    for i, (xi, yi) in enumerate(zip(X, y)):
        dot  = np.dot(w, xi)
        pred = +1 if dot > 0 else -1
        if pred != yi:
            w += eta * yi * xi
            errors += 1
            print(f"{epoch:>6} {i+1:>4} {yi:>6} {dot:>8.2f} {pred:>6}  "
                  f"[{w[0]:+.2f},{w[1]:+.2f},{w[2]:+.2f}]")
    if errors == 0:
        print(f"\nConverged after epoch {epoch}!")
        break

print(f"\nFinal weights: {w}")
print(f"Decision: {w[1]:.2f}*x1 + {w[2]:.2f}*x2 + {w[0]:.2f} = 0\n")
print("Verification:")
for xi, yi in zip(X, y):
    dot  = np.dot(w, xi)
    pred = +1 if dot > 0 else -1
    ok   = "OK" if pred == yi else "WRONG"
    print(f"  ({xi[1]:+d},{xi[2]:+d}) -> w.x={dot:.2f} pred={pred:+d} true={yi:+d} {ok}")

=== Perceptron Learning: Step-by-Step ===

Initial weights: [0. 0. 0.]
Learning rate η: 0.5

 Epoch   Ex   True      w.x   Pred  New weights
-----------------------------------------------------------------
     1    1      1     0.00     -1  [+0.50,+0.50,+0.50]

Converged after epoch 2!

Final weights: [0.5 0.5 0.5]
Decision: 0.50*x1 + 0.50*x2 + 0.50 = 0

Verification:
  (+1,+1) -> w.x=1.50 pred=+1 true=+1 OK
  (+2,+1) -> w.x=2.00 pred=+1 true=+1 OK
  (+1,+2) -> w.x=2.00 pred=+1 true=+1 OK
  (-1,-1) -> w.x=-0.50 pred=-1 true=-1 OK
  (-2,-1) -> w.x=-1.00 pred=-1 true=-1 OK
  (-1,-2) -> w.x=-1.00 pred=-1 true=-1 OK


## Example 3: MLFFNN Solving XOR + Architecture Comparison

XOR is the canonical example showing why we need hidden layers. A single perceptron (no hidden layer) gets only 25% accuracy on XOR — it literally cannot draw a line that separates the two classes. Adding one hidden layer of 4 tanh neurons solves it instantly: 100% accuracy. We then print out all the learned weights to make the network transparent — you can trace how input (0,1) flows through the network and comes out as 1. The final section compares three architecture sizes on a real 3-class problem and shows how adding more parameters doesn't always improve test accuracy (due to overfitting on a small dataset).

In [3]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

print("=== XOR Problem ===\n")
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0, 1, 1, 0])

# Perceptron: no hidden layers
perc = MLPClassifier(hidden_layer_sizes=(), activation='logistic',
                     max_iter=1000, random_state=42)
perc.fit(X_xor, y_xor)
print(f"Perceptron (0 hidden): acc={accuracy_score(y_xor, perc.predict(X_xor))*100:.0f}%  preds={perc.predict(X_xor)}")

# MLFFNN: 1 hidden layer
mlp = MLPClassifier(hidden_layer_sizes=(4,), activation='tanh',
                    max_iter=3000, random_state=1)
mlp.fit(X_xor, y_xor)
print(f"MLFFNN (4 hidden)    : acc={accuracy_score(y_xor, mlp.predict(X_xor))*100:.0f}%  preds={mlp.predict(X_xor)}")

print("\nLearned hidden-layer weights (2 inputs -> 4 neurons):")
for j in range(4):
    w1, w2 = mlp.coefs_[0][0,j], mlp.coefs_[0][1,j]
    b = mlp.intercepts_[0][j]
    print(f"  Neuron {j+1}: x1*{w1:+.3f} + x2*{w2:+.3f} + bias:{b:+.3f}")

print("\n=== Architecture Comparison (3-class task) ===\n")
np.random.seed(42)
X2, y2 = make_classification(n_samples=1000, n_features=10, n_informative=7,
                              n_classes=3, n_clusters_per_class=1, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X2, y2, test_size=0.2, random_state=42)
print(f"{'Architecture':<20} {'Test Acc':>10} {'Total Params':>14}")
print("-" * 48)
for arch in [(10,), (20,10), (50,30,10)]:
    m = MLPClassifier(hidden_layer_sizes=arch, activation='relu',
                      max_iter=600, random_state=42)
    m.fit(Xtr, ytr)
    acc    = accuracy_score(yte, m.predict(Xte))
    params = sum(w.size for w in m.coefs_) + sum(b.size for b in m.intercepts_)
    print(f"  {str(arch):<18} {acc*100:>9.1f}% {params:>14}")

=== XOR Problem ===

Perceptron (0 hidden): acc=25%  preds=[1 1 0 1]
MLFFNN (4 hidden)    : acc=100%  preds=[0 1 1 0]

Learned hidden-layer weights (2 inputs -> 4 neurons):
  Neuron 1: x1*-1.584 + x2*-2.045 + bias:+0.086
  Neuron 2: x1*+1.380 + x2*-2.327 + bias:-0.567
  Neuron 3: x1*-2.385 + x2*-1.930 + bias:+0.320
  Neuron 4: x1*-2.277 + x2*+1.388 + bias:-0.451

=== Architecture Comparison (3-class task) ===

Architecture           Test Acc   Total Params
------------------------------------------------
  (10,)                   94.0%            143
  (20, 10)                92.5%            463
  (50, 30, 10)            93.5%           2423
